In [0]:
from pyspark import pipelines as dp

## Store the target configuration environment in the variable targert
target = spark.conf.get("target")

### Create a Dictionary for Integration Test Values

Create a dictionary containing the necessary values for integration tests in both **development** and **stage** environments. There are several approaches to achieve this, but this is a straightforward method.

For more information, refer to the [Portable and Reusable Expectations](https://docs.databricks.com/en/delta-live-tables/expectation-patterns.html#portable-and-reusable-expectations) documentation.



In [0]:
## Based on the deployed target, obtain the specific validation metrics for the tables.
target_integration_tests_validation = {
    'development': {
        'health_bronze': {
            'total_rows': 7500
        },
        'health_silver': {
            'total_rows': 7500
        }
    },
    'stage': {
        'health_bronze': {
            'total_rows': 35000
        },
        'health_silver': {
            'total_rows': 35000
        }
    }
}


## Store the expected values for the total rows in the tables tables in the variables based on the target if in development or stage
if target in ('development', 'stage'):
    total_expected_bronze = target_integration_tests_validation[target]['health_bronze']['total_rows']
    total_expected_silver = target_integration_tests_validation[target]['health_silver']['total_rows']

### Create a Function to Count the Total Number of Rows in a Table
The `test_count_table_total_rows` function creates a materialized view that counts the total number of rows in the specified table.

In [0]:
def test_count_table_total_rows(table_name, total_count, target):
    '''
    Count the number of rows in the specified table and compare with the expected values for development and stage data. 
    Fail the update if the count does not match the specified values.
    '''
    # Creates a materialized view (a table) in your pipeline with the specified name and comment
    @dp.table( 
        name=f"TEST_{target}_{table_name}_total_rows_verification",
        comment=f"Confirms all rows were ingested from the {target} raw data to {table_name}"
    )

    #  A data quality check that validates every row. If any row fails the check (total_rows = {total_count}), the entire pipeline update will fail and stop
    @dp.expect_all_or_fail({"valid count": f"total_rows = {total_count}"}) 

    # When the pipeline runs, it executes the inner function (registers and create the actual dataset), which counts rows in 'table_name' and validates the count equals 'total_count'
    def count_table_total_rows():
        return spark.sql(f"""
            SELECT COUNT(*) AS total_rows FROM {table_name}
        """)

### Create a Function to Confirm the Column Values in the Gold Materialized View
The `test_gold_table_columns` function creates a materialized view that checks the values in the columns **Age_Group** and **HighCholest_Group** in **chol_age_agg**.

In [0]:
def test_gold_table_columns():
    '''
    This function will check unique values in the columns Age_Group and HighCholest_Group in the gold table chol_age_agg.

    This confirms that the distinct values for these columns in the gold table are correct.
    ''' 
    ## Set expectations for the columns
    check_silver_calc_columns = {
        "valid age group": "Age_Group in ('0-9', '10-19', '20-29', '30-39', '40-49', '50+', 'Unknown')",
        "valid cholest group": "HighCholest_Group in ('Normal', 'Above Average', 'High', 'Unknown')"
    }

    @dp.table(comment="Check age group and high cholest group in the gold table")

    ## Fail if expectations are not met
    @dp.expect_all_or_fail(check_silver_calc_columns)

    def test_calculated_columns_age_cholesterol():
        return (dp
                .read("chol_age_agg")
                .select("Age_Group", "HighCholest_Group")
            )

### Execute the Specified Integration Tests
Execute the specified integration tests based on the target environment.

In [0]:
## Run the specified tests based on the target environment (development, stage or production)

if target in ('development','stage'):  ## Dynamic integration test for dev or stage tables
    test_count_table_total_rows('health_bronze',  total_expected_bronze, target)
    test_count_table_total_rows('health_silver',  total_expected_silver, target)
    test_gold_table_columns()
elif target == 'production':  ## Only test the gold table in production
    test_gold_table_columns()

## Option 2 - Integration Testing with Notebooks and Databricks Jobs
You can also perform integration testing using notebooks and add them as tasks in jobs for your pipeline. 

#### Steps to take:
1. Create a setup notebook to handle any dynamic setup required using job parameters for your target environment and data locations.

2. Create additional notebooks or files to store the integration tests you want to run as tasks.

3. Organize the new notebooks or files within your **tests** folder.

4. Create a Workflow. Within the Workflow:

   - a. Create the necessary tables or views using Spark Declarative Pipeline or code.

   - b. Add tasks to set up your integration tests (e.g., setting up any dynamic job parameters that need to be set).

   - c. Perform validation by using your notebooks as tasks and set the tasks to all should succeed.

**NOTES:** One major drawback of this approach is that you will need to write more code for setup and validation tasks, as well as manage the job parameters to dynamically modify the code based on the target environment.